# Calculadora Von Neumann optimizada — Interfaz para Colab

Ejecuta la celda de abajo (Shift+Enter). Usa solo `dataclasses` (estándar de Python) e `ipywidgets`, que viene preinstalado en Google Colab — no requiere instalar nada.

In [1]:
# Calculadora Harvard optimizada - Interfaz interactiva para Colab
# Librerias usadas: ipywidgets (preinstalada en Colab), IPython.display

import ipywidgets as widgets
from IPython.display import display, clear_output


# ---------------- MODULO 1: MEMORIA DE INSTRUCCIONES ----------------
class MemoriaInstrucciones:
    def __init__(self):
        self.instrucciones = {
            "1": "SUMA",
            "2": "RESTA",
            "3": "MULTIPLICACION",
            "4": "DIVISION",
        }

    def obtener_nombre(self, opcode):
        return self.instrucciones.get(opcode)


# ---------------- MODULO 2: MEMORIA DE DATOS ----------------
class MemoriaDatos:
    def __init__(self):
        self.historial = []

    def guardar(self, valor):
        direccion = len(self.historial)
        self.historial.append(valor)
        return direccion

    def ver_historial(self):
        return list(self.historial)


# ---------------- MODULO 3: ALU ----------------
class ALU:
    @staticmethod
    def calcular(nombre_operacion, a, b):
        if nombre_operacion == "SUMA":
            return a + b
        if nombre_operacion == "RESTA":
            return a - b
        if nombre_operacion == "MULTIPLICACION":
            return a * b
        if nombre_operacion == "DIVISION":
            if b == 0:
                raise ZeroDivisionError("La ALU no puede dividir entre cero")
            return a / b
        raise ValueError(f"La ALU no reconoce la operacion: {nombre_operacion}")


# ---------------- MODULO 4: UNIDAD DE CONTROL ----------------
class UnidadDeControl:
    def __init__(self, memoria_instrucciones, alu):
        self.memoria_instrucciones = memoria_instrucciones
        self.alu = alu

    def decodificar_y_ejecutar(self, opcode, a, b):
        nombre_operacion = self.memoria_instrucciones.obtener_nombre(opcode)
        if nombre_operacion is None:
            raise ValueError(f"Instruccion no valida: {opcode!r}")
        resultado = self.alu.calcular(nombre_operacion, a, b)
        return nombre_operacion, resultado


# ---------------- MODULO 5: CPU ----------------
class CPU:
    def __init__(self):
        self.memoria_instrucciones = MemoriaInstrucciones()
        self.memoria_datos = MemoriaDatos()
        self.alu = ALU()
        self.control = UnidadDeControl(self.memoria_instrucciones, self.alu)

    def ejecutar_ciclo(self, opcode, a, b, log_callback):
        log_callback("--- CICLO DE INSTRUCCION ---")
        try:
            nombre_operacion, resultado = self.control.decodificar_y_ejecutar(opcode, a, b)
        except ZeroDivisionError as err:
            log_callback(f"ERROR en la ALU: {err}")
            return None

        log_callback(f"DECODE/EXECUTE -> operacion: {nombre_operacion}")
        direccion = self.memoria_datos.guardar(resultado)
        log_callback(f"WRITE-BACK     -> resultado {resultado} guardado en memoria_datos[{direccion}]")
        return resultado



# INTERFAZ (widgets)

titulo = widgets.HTML("<h2>Calculadora - Modelo optimizado (arquitectura Harvard)</h2>")

entrada_a = widgets.FloatText(description="Operando A:", value=0)
entrada_b = widgets.FloatText(description="Operando B:", value=0)
operacion = widgets.Dropdown(
    options=[("Sumar (+)", "1"), ("Restar (-)", "2"),
             ("Multiplicar (x)", "3"), ("Dividir (/)", "4")],
    description="Instruccion:",
)

boton_ejecutar = widgets.Button(description="Ejecutar ciclo", button_style="success")
boton_limpiar = widgets.Button(description="Limpiar consola", button_style="warning")

panel_modulos = widgets.HTML()
consola = widgets.Output(layout=widgets.Layout(
    border="1px solid black", height="180px", overflow_y="auto"
))
panel_memoria_datos = widgets.Output(layout=widgets.Layout(
    border="1px solid gray", height="100px", overflow_y="auto"
))


def actualizar_panel_modulos():
    panel_modulos.value = (
        "<b>Modulos activos</b><br>"
        f"Memoria de Instrucciones: {list(cpu.memoria_instrucciones.instrucciones.values())}<br>"
        f"Memoria de Datos: {len(cpu.memoria_datos.historial)} resultado(s) guardado(s)<br>"
        "ALU: lista para calcular<br>"
        "Unidad de Control: coordinando modulos"
    )


def actualizar_memoria_datos():
    with panel_memoria_datos:
        clear_output()
        for i, valor in enumerate(cpu.memoria_datos.ver_historial()):
            print(f"memoria_datos[{i}] = {valor}")


def log_en_consola(texto):
    with consola:
        print(texto)


def al_hacer_click_ejecutar(_):
    cpu.ejecutar_ciclo(operacion.value, entrada_a.value, entrada_b.value, log_en_consola)
    actualizar_panel_modulos()
    actualizar_memoria_datos()


def al_hacer_click_limpiar(_):
    with consola:
        clear_output()


boton_ejecutar.on_click(al_hacer_click_ejecutar)
boton_limpiar.on_click(al_hacer_click_limpiar)

actualizar_panel_modulos()

entradas = widgets.VBox([entrada_a, entrada_b, operacion,
                          widgets.HBox([boton_ejecutar, boton_limpiar]),
                          panel_modulos])
salida = widgets.VBox([widgets.HTML("<b>Ciclo Fetch-Decode-Execute-WriteBack</b>"), consola,
                        widgets.HTML("<b>Memoria de Datos (historial)</b>"), panel_memoria_datos])

display(titulo, widgets.HBox([entradas, salida]))

NameError: name 'cpu' is not defined